# Avancement — couche Silver

Ce notebook rend compte de deux points de la liste *Reste à faire* du README :

| Tâche | État |
|---|---|
| `conf/silver_mapping.yml` — mapping champs source → modèle commun | **fait** |
| `src/silver/` — validation de schéma, dédup, normalisation UTC, dépivotement des filières | **fait** |
| `src/gold/`, `dags/`, `notebooks/`, `src/ml/` | déjà présents dans le dépôt |

Il ne se contente pas de décrire : chaque affirmation est **exécutée** ci‑dessous,
sur les schémas réels des API et sur un Bronze factice qui respecte
l'arborescence de production. Aucun cluster, aucun HDFS nécessaire.

> Restitution métier depuis Gold : voir `notebooks/insights.ipynb`.
> Ce notebook‑ci porte sur l'état d'avancement et les choix de conception.

In [1]:
import os, sys, json, pathlib, tempfile, warnings
warnings.filterwarnings("ignore")

# Fonctionne aussi bien dans le conteneur (/opt/datalake) qu'en local.
def repo_root() -> pathlib.Path:
    for base in [pathlib.Path("/opt/datalake"), *pathlib.Path.cwd().parents,
                 pathlib.Path.cwd()]:
        if (base / "conf" / "silver_mapping.yml").exists():
            return base
    raise RuntimeError("conf/silver_mapping.yml introuvable")

ROOT = repo_root()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
os.environ.setdefault("DATALAKE_SILVER_MAPPING",
                      str(ROOT / "conf" / "silver_mapping.yml"))
# Evite l'avertissement Spark sur la resolution du nom d'hote en local.
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

import pandas as pd
pd.set_option("display.max_columns", 40, "display.width", 160)
print("Dépôt :", ROOT)

Dépôt : /Users/theo-dev/Dev/projet_final


## 1. Le mapping, colonne vertébrale de la couche

`src/silver/` ne contient **aucun nom de champ source en dur**. Tout est déclaré
dans `conf/silver_mapping.yml`, ce qui veut dire qu'ajouter une filière ou
renommer une colonne côté RTE se règle en éditant un YAML.

Le module `silver.mapping` n'importe pas pyspark : il se charge depuis un
notebook ou un script de dev, sans session Spark.

In [2]:
from silver.mapping import load_mapping, normalize

M = load_mapping()
print(f"mapping v{M.version} — {len(M.tables)} tables, "
      f"{len(M.sources)} sources, {len(M.filieres)} filières déclarées\n")

rows = []
for name, t in M.tables.items():
    rows.append({
        "table": name,
        "clés": ", ".join(t.keys),
        "dédup": t.dedup_strategy,
        "mesures": len(t.measures) or "dépivot",
        "contrôles": len(t.not_null) + len(t.dimensions),
    })
display(pd.DataFrame(rows))

# Le contrat d'une table, mesure par mesure.
mes = [{"colonne": k, "champs source acceptés": ", ".join(m.candidates),
        "type": m.dtype, "unité": m.unit,
        "min": m.minimum, "max": m.maximum,
        "hors bornes": m.on_range_violation}
       for k, m in M.table("grid_load").measures.items()]
display(pd.DataFrame(mes))

mapping v1 — 3 tables, 3 sources, 27 filières déclarées



,table,clés,dédup,mesures,contrôles
0,grid_load,"ts_utc, zone_id",merge,5,2
1,grid_generation,"ts_utc, zone_id, filiere",priority,dépivot,2
2,weather,"ts_utc, zone_id",priority,4,2


,colonne,champs source acceptés,type,unité,min,max,hors bornes
0,consumption_mw,"consommation, consommation_mw, conso",double,MW,10000,120000,reject
1,forecast_j1_mw,"prevision_j1, prevision_j_1, previsionj1",double,MW,10000,120000,null_out
2,forecast_j_mw,"prevision_j, previsionj",double,MW,10000,120000,null_out
3,co2_rate_g_kwh,"taux_co2, taux_de_co2, co2",double,gCO2/kWh,0,1200,null_out
4,physical_exchange_mw,"ech_physiques, echanges_physiques",double,MW,-25000,25000,null_out


La résolution des noms est insensible à la casse, aux accents et à la
ponctuation. C'est ce qui permet de survivre à un export qui basculerait des
noms techniques aux libellés.

In [3]:
for s in ["Consommation (MW)", "DATE-HEURE", "Données définitives", "Éolien offshore"]:
    print(f"  {s!r:26} -> {normalize(s)!r}")

  'Consommation (MW)'        -> 'consommation_mw'
  'DATE-HEURE'               -> 'date_heure'
  'Données définitives'      -> 'donnees_definitives'
  'Éolien offshore'          -> 'eolien_offshore'


## 2. La hiérarchie des filières, et le double compte qu'elle évite

`eolien` **vaut déjà** `eolien_terrestre + eolien_offshore` : les trois sont
publiées côte à côte par RTE. Les dépivoter toutes ferait double compte dès que
Gold somme la production totale.

Le mapping range donc chaque filière sur un niveau (`aggregate` / `detail`) et
dans une catégorie (`production` / `stockage`), et n'émet par défaut que les
agrégats. Le détail reste déclaré, activable via `unpivot.include_levels`.

In [4]:
fil = pd.DataFrame([{"filiere": f.name, "niveau": f.level, "catégorie": f.category,
                     "renouvelable": f.renewable, "parent": f.parent or ""}
                    for f in M.filieres.values()])
print(fil.groupby(["niveau", "catégorie"]).size().to_string(), "\n")
display(fil[fil.niveau == "aggregate"].reset_index(drop=True))
print("\nDétail exclu par défaut (sinon double compte) :")
print(", ".join(fil[fil.niveau == "detail"].filiere))

niveau     catégorie 
aggregate  production     9
           stockage       3
detail     production    15 



,filiere,niveau,catégorie,renouvelable,parent
0,nucleaire,aggregate,production,False,
1,thermique,aggregate,production,False,
2,charbon,aggregate,production,False,
3,fioul,aggregate,production,False,
4,gaz,aggregate,production,False,
5,eolien,aggregate,production,True,
6,solaire,aggregate,production,True,
7,hydraulique,aggregate,production,True,
8,bioenergies,aggregate,production,True,
9,pompage,aggregate,stockage,False,



Détail exclu par défaut (sinon double compte) :
eolien_terrestre, eolien_offshore, fioul_tac, fioul_cogen, fioul_autres, gaz_tac, gaz_cogen, gaz_ccg, gaz_autres, hydraulique_fil_eau_eclusee, hydraulique_lacs, hydraulique_step_turbinage, bioenergies_dechets, bioenergies_biomasse, bioenergies_biogaz


## 3. Confrontation aux schémas réels des API

Les noms de champs éCO2mix évoluent. Le README insiste : **ne pas les figer
depuis une doc**, les lire sur l'API. C'est fait ici en direct — et si le
notebook tourne hors ligne, on retombe sur les schémas relevés le 2026‑08‑27.

In [5]:
import urllib.request

API = ("https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/"
       "{ds}/records?limit=1")
DATASETS = {"eco2mix_tr": "eco2mix-national-tr",
            "eco2mix_cons": "eco2mix-national-cons-def"}
# Relevé du 2026-08-27, utilisé si l'API n'est pas joignable.
FALLBACK = {
    "eco2mix_tr": ["perimetre","nature","date","heure","date_heure","consommation",
        "prevision_j1","prevision_j","fioul","charbon","gaz","nucleaire","eolien",
        "eolien_terrestre","eolien_offshore","solaire","hydraulique","pompage",
        "bioenergies","ech_physiques","taux_co2","ech_comm_angleterre",
        "ech_comm_espagne","ech_comm_italie","ech_comm_suisse",
        "ech_comm_allemagne_belgique","fioul_tac","fioul_cogen","fioul_autres",
        "gaz_tac","gaz_cogen","gaz_ccg","gaz_autres","hydraulique_fil_eau_eclusee",
        "hydraulique_lacs","hydraulique_step_turbinage","bioenergies_dechets",
        "bioenergies_biomasse","bioenergies_biogaz","stockage_batterie",
        "destockage_batterie"],
}
FALLBACK["eco2mix_cons"] = [c for c in FALLBACK["eco2mix_tr"]
                            if c not in ("eolien_terrestre","eolien_offshore",
                                         "stockage_batterie","destockage_batterie")]

def fields(key):
    try:
        with urllib.request.urlopen(API.format(ds=DATASETS[key]), timeout=20) as r:
            rec = json.load(r)["results"][0]
        return list(rec), "API"
    except Exception as exc:
        print(f"  ({key} : API injoignable — {type(exc).__name__}, relevé local)")
        return FALLBACK[key], "relevé"

rows = []
for key in DATASETS:
    cols, origine = fields(key)
    found_m, missing_m = M.resolve_measures("grid_load", cols)
    found_f, missing_f = M.resolve_filieres(cols)
    rows.append({"source": key, "origine": origine, "champs": len(cols),
                 "mesures résolues": f"{len(found_m)}/{len(M.table('grid_load').measures)}",
                 "filières trouvées": len(found_f),
                 "filières absentes": ", ".join(missing_f) or "—"})
display(pd.DataFrame(rows))

,source,origine,champs,mesures résolues,filières trouvées,filières absentes
0,eco2mix_tr,API,41,5/5,11,thermique
1,eco2mix_cons,API,37,5/5,9,"thermique, stockage_batterie, destockage_batterie"


Les deux flux **n'ont pas le même schéma** : 41 champs en temps réel, 37 en
consolidé. `eolien_offshore` et `stockage_batterie` n'existent que d'un côté.
Un champ absent est signalé puis mis à `null` — jamais fatal.

Deux autres écarts, invisibles sans ce travail de mapping :

- `ech_comm_allemagne_belgique` arrive en **entier** d'un flux et en **chaîne**
  de l'autre → le typage doit être explicite, jamais inféré.
- Le jeu *définitif* ne publie les mesures qu'au pas de **30 min**, alors que
  les prévisions sont au quart d'heure. Une ligne sur deux n'a que des
  prévisions : ce n'est pas une anomalie, c'est le format. D'où la stratégie de
  fusion mesure par mesure, démontrée plus bas.

## 4. Les quatre opérations, exécutées

On fabrique un Bronze factice qui respecte l'arborescence réelle
(`year=/month=`, `ingest_date=/ingest_hour=`, `city=`) et on lance dessus les
vrais lecteurs et les vraies transformations — le même code que celui
soumis par Airflow.

In [6]:
from pyspark.sql import SparkSession, functions as F
import test_silver_local as fixtures

TMP = pathlib.Path(tempfile.mkdtemp(prefix="silver_nb_"))
fixtures.build_bronze(TMP)

spark = (SparkSession.builder.appName("avancement-silver").master("local[2]")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
         .config("spark.ui.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

for p in sorted(TMP.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(TMP))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/27 14:36:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


   bronze/eco2mix_cons/year=2024/month=03/eco2mix_national_2024_03.csv
   bronze/eco2mix_tr/ingest_date=2024-03-15/ingest_hour=01/part-00000.json
   bronze/meteo_archive/city=lyon/year=2024/month=03/open_meteo_lyon_2024_03.json
   bronze/meteo_archive/city=paris/year=2024/month=03/open_meteo_paris_2024_03.json


### 4.1 Lecture Bronze et normalisation UTC

Trois formats, trois lecteurs, un seul point d'entrée piloté par le mapping.

In [7]:
from silver.readers import read_source
from silver.silver_grid import prepare, build_grid_load, build_grid_generation

raw_cons = read_source(spark, M, "eco2mix_cons", "", str(TMP))
raw_tr   = read_source(spark, M, "eco2mix_tr",   "", str(TMP))
print(f"CSV consolidé : {len(raw_cons.columns)} colonnes, {raw_cons.count()} lignes")
print(f"Streaming     : {len(raw_tr.columns)} colonnes (payload JSON parsé en Silver)")

cons = prepare(raw_cons, M, "eco2mix_cons")
(cons.select(F.date_format("ts_utc", "yyyy-MM-dd HH:mm").alias("ts_utc"),
             F.date_format("ts_local", "yyyy-MM-dd HH:mm").alias("ts_local"),
             "quality", "consommation")
     .orderBy("ts_utc").show(4, truncate=False))

2026-08-27 14:36:14,361 INFO    [silver-grid] Lecture eco2mix_cons (csv) depuis /var/folders/dj/9szbc9yn265_gn9_ktgw85mr0000gn/T/silver_nb_k814vyp6/bronze/eco2mix_cons/year=*/month=*/*.csv


2026-08-27 14:36:20,706 INFO    [silver-grid] eco2mix_cons : 21 colonne(s) detectee(s).


2026-08-27 14:36:20,764 INFO    [silver-grid] Lecture eco2mix_tr (json_envelope) depuis /var/folders/dj/9szbc9yn265_gn9_ktgw85mr0000gn/T/silver_nb_k814vyp6/bronze/eco2mix_tr/ingest_date=*/ingest_hour=*/*


2026-08-27 14:36:23,434 INFO    [silver-grid] eco2mix_tr : 23 colonne(s) apres parsing du payload.


2026-08-27 14:36:24,530 INFO    [silver-grid] Colonne temporelle : date_heure (type timestamp)


CSV consolidé : 22 colonnes, 111 lignes
Streaming     : 23 colonnes (payload JSON parsé en Silver)


2026-08-27 14:36:24,802 INFO    [silver-grid] Qualite lue dans le champ 'nature' (defaut : consolidated).


+----------------+----------------+----------+------------+
|ts_utc          |ts_local        |quality   |consommation|
+----------------+----------------+----------+------------+
|NULL            |NULL            |definitive|52000       |
|2024-03-14 23:00|2024-03-15 00:00|definitive|50000       |
|2024-03-14 23:15|2024-03-15 00:15|definitive|50010       |
|2024-03-14 23:30|2024-03-15 00:30|definitive|50020       |
+----------------+----------------+----------+------------+
only showing top 4 rows



`00:00` locale devient `23:00` la veille en UTC : l'offset ISO est lu, pas
supposé. Et la qualité est lue **dans** la donnée (champ `nature`), pas déduite
du chemin Bronze — un lot mensuel peut être à cheval sur la frontière
consolidé / définitif.

Le cas qui casse la plupart des pipelines, la nuit du changement d'heure :
`02:00` locale existe **deux fois** le 27 octobre.

In [8]:
dst = cons.filter(F.col("ts_local").cast("string").startswith("2024-10-27 02:"))
(dst.select(F.date_format("ts_utc", "yyyy-MM-dd HH:mm").alias("ts_utc"),
            F.date_format("ts_local", "yyyy-MM-dd HH:mm").alias("ts_local"),
            "consommation")
    .orderBy("ts_utc").show(8, truncate=False))
print(f"{dst.count()} lignes sur 02:xx locale, "
      f"{dst.select('ts_utc').distinct().count()} horodatages UTC distincts "
      "→ aucun quart d'heure écrasé.")

+----------------+----------------+------------+
|ts_utc          |ts_local        |consommation|
+----------------+----------------+------------+
|2024-10-27 00:00|2024-10-27 02:00|41000       |
|2024-10-27 00:15|2024-10-27 02:15|41000       |
|2024-10-27 00:30|2024-10-27 02:30|41000       |
|2024-10-27 00:45|2024-10-27 02:45|41000       |
|2024-10-27 01:00|2024-10-27 02:00|42000       |
|2024-10-27 01:15|2024-10-27 02:15|42000       |
|2024-10-27 01:30|2024-10-27 02:30|42000       |
|2024-10-27 01:45|2024-10-27 02:45|42000       |
+----------------+----------------+------------+



8 lignes sur 02:xx locale, 8 horodatages UTC distincts → aucun quart d'heure écrasé.


### 4.2 Validation de schéma et quarantaine

Le jeu factice contient quatre lignes fautives, une par motif. Aucune n'est
supprimée en silence : toutes partent dans `/datalake/silver/_rejects/` avec
leur charge utile complète, rejouables après correction du mapping.

In [9]:
from silver.transform import restrict_window

win = restrict_window(cons, "2024-01-01", "2024-12-31", keep_null=True)
load_cons = build_grid_load(win, M, "eco2mix_cons")

print(json.dumps({k: v for k, v in load_cons.report.as_dict().items()
                  if k in ("n_input", "n_valid", "n_rejected",
                           "n_dropped_empty", "reject_ratio", "reasons")},
                 indent=2, ensure_ascii=False))

load_cons.rejects.select("reject_reason", "reject_column", "source",
                         F.substring("payload", 1, 78).alias("payload (extrait)")) \
                 .show(truncate=False)

{
  "n_input": 111,
  "n_valid": 105,
  "n_rejected": 4,
  "n_dropped_empty": 2,
  "reject_ratio": 0.036036,
  "reasons": {
    "unexpected_value:perimeter": 1,
    "cast_failed:consumption_mw": 1,
    "out_of_range:consumption_mw": 1,
    "null_key:ts_utc": 1
  }
}


+----------------+--------------+------------+------------------------------------------------------------------------------+
|reject_reason   |reject_column |source      |payload (extrait)                                                             |
+----------------+--------------+------------+------------------------------------------------------------------------------+
|out_of_range    |consumption_mw|eco2mix_cons|{"perimetre":"France","nature":"Donnees definitives","date":"2024-03-17","heur|
|cast_failed     |consumption_mw|eco2mix_cons|{"perimetre":"France","nature":"Donnees definitives","date":"2024-03-17","heur|
|unexpected_value|perimeter     |eco2mix_cons|{"perimetre":"Grand-Est","nature":"Donnees definitives","date":"2024-03-17","h|
|null_key        |ts_utc        |eco2mix_cons|{"perimetre":"France","nature":"Donnees definitives","consommation":"52000","p|
+----------------+--------------+------------+------------------------------------------------------------------------

Les deux lignes « vides » ne sont **pas** des rejets : le définitif ne publie
les mesures qu'au pas de 30 min, une ligne sans mesure est le format normal.
Elles sont écartées et comptées à part.

Et une prévision aberrante n'emporte pas la consommation mesurée de la même
ligne : `on_range_violation: null_out` neutralise la seule mesure fautive.

In [10]:
(load_cons.df
 .filter(F.col("consumption_mw") == 53000)
 .select(F.date_format("ts_utc", "yyyy-MM-dd HH:mm").alias("ts_utc"),
         "consumption_mw", "forecast_j1_mw", "co2_rate_g_kwh")
 .show(truncate=False))
print("prevision_j1 valait 999999 à la source : mesure annulée, ligne conservée.")

+----------------+--------------+--------------+--------------+
|ts_utc          |consumption_mw|forecast_j1_mw|co2_rate_g_kwh|
+----------------+--------------+--------------+--------------+
|2024-03-17 00:00|53000.0       |NULL          |45.0          |
+----------------+--------------+--------------+--------------+

prevision_j1 valait 999999 à la source : mesure annulée, ligne conservée.


### 4.3 Déduplication : le consolidé gagne, mais mesure par mesure

Le temps réel et le consolidé se recouvrent volontairement sur `2024-03-15`.
La valeur temps réel (`99999`, bidon) doit disparaître. Mais à `04:00 UTC`, le
consolidé n'a **pas** de taux de CO₂ et le temps réel en a un : une dédup
naïve, ligne par ligne, perdrait cette mesure.

In [11]:
from silver.transform import dedupe

load_tr = build_grid_load(
    restrict_window(prepare(raw_tr, M, "eco2mix_tr"), "2024-01-01", "2024-12-31",
                    keep_null=True), M, "eco2mix_tr")

spec = M.table("grid_load")
union = load_cons.df.unionByName(load_tr.df)
load = dedupe(union, list(spec.keys), spec.dedup_strategy, list(spec.measures))

print(f"{union.count()} lignes -> {load.count()} après déduplication")
print(f"valeurs temps réel survivantes (99999) : "
      f"{load.filter(F.col('consumption_mw') == 99999).count()}")

(load.filter(F.col("ts_utc") == F.lit("2024-03-15 04:00:00").cast("timestamp"))
     .select(F.date_format("ts_utc", "yyyy-MM-dd HH:mm").alias("ts_utc"),
             "consumption_mw", "co2_rate_g_kwh", "source", "quality")
     .show(truncate=False))
print("consommation issue du définitif, taux CO2 récupéré sur le temps réel.")

2026-08-27 14:36:36,723 INFO    [silver-grid] Colonne temporelle : date_heure (type string)


2026-08-27 14:36:36,746 INFO    [silver-grid] Colonne texte : offset lu s'il est present, sinon Europe/Paris.


2026-08-27 14:36:36,896 INFO    [silver-grid] Qualite lue dans le champ 'nature' (defaut : realtime).


114 lignes -> 105 après déduplication


valeurs temps réel survivantes (99999) : 0


+----------------+--------------+--------------+------------+----------+
|ts_utc          |consumption_mw|co2_rate_g_kwh|source      |quality   |
+----------------+--------------+--------------+------------+----------+
|2024-03-15 04:00|50200.0       |42.0          |eco2mix_cons|definitive|
+----------------+--------------+--------------+------------+----------+

consommation issue du définitif, taux CO2 récupéré sur le temps réel.


### 4.4 Dépivotement des filières

Une colonne par filière devient une ligne par filière. Le schéma n'a plus à
bouger quand RTE ajoute une filière — et chaque ligne porte sa catégorie et son
niveau, pour que Gold puisse sommer sans double compte.

In [12]:
gen_cons = build_grid_generation(win, M, "eco2mix_cons")
gen_tr = build_grid_generation(
    restrict_window(prepare(raw_tr, M, "eco2mix_tr"), "2024-01-01", "2024-12-31",
                    keep_null=True), M, "eco2mix_tr")
gen = gen_cons.df.unionByName(gen_tr.df)

(gen.groupBy("filiere", "filiere_category", "filiere_level", "is_renewable")
    .count().orderBy("filiere_category", "filiere").show(20, truncate=False))

print("filières déclarées mais absentes du consolidé :",
      ", ".join(gen_cons.report.missing_filieres))
print("émises depuis le temps réel :", len(gen_tr.report.present_filieres))
print("lignes de niveau 'detail' émises :",
      gen.filter(F.col("filiere_level") != "aggregate").count(),
      "→ pas de double compte possible en Gold")

2026-08-27 14:36:45,840 INFO    [silver-grid] 9 filiere(s) depivotee(s) : bioenergies, charbon, eolien, fioul, gaz, hydraulique, nucleaire, pompage, solaire


2026-08-27 14:36:47,981 INFO    [silver-grid] Colonne temporelle : date_heure (type string)


2026-08-27 14:36:47,998 INFO    [silver-grid] Colonne texte : offset lu s'il est present, sinon Europe/Paris.


2026-08-27 14:36:48,122 INFO    [silver-grid] Qualite lue dans le champ 'nature' (defaut : realtime).


2026-08-27 14:36:48,304 INFO    [silver-grid] 11 filiere(s) depivotee(s) : bioenergies, charbon, destockage_batterie, eolien, fioul, gaz, hydraulique, nucleaire, pompage, solaire, stockage_batterie


+-------------------+----------------+-------------+------------+-----+
|filiere            |filiere_category|filiere_level|is_renewable|count|
+-------------------+----------------+-------------+------------+-----+
|bioenergies        |production      |aggregate    |true        |116  |
|charbon            |production      |aggregate    |false       |116  |
|eolien             |production      |aggregate    |true        |117  |
|fioul              |production      |aggregate    |false       |116  |
|gaz                |production      |aggregate    |false       |116  |
|hydraulique        |production      |aggregate    |true        |116  |
|nucleaire          |production      |aggregate    |false       |117  |
|solaire            |production      |aggregate    |true        |116  |
|destockage_batterie|stockage        |aggregate    |false       |8    |
|pompage            |stockage        |aggregate    |false       |116  |
|stockage_batterie  |stockage        |aggregate    |false       

lignes de niveau 'detail' émises : 0 → pas de double compte possible en Gold


### 4.5 Météo : tableaux parallèles, unités, moyenne pondérée

Open‑Meteo renvoie `hourly.time[i]` / `hourly.temperature_2m[i]` : des tableaux
parallèles, pas des lignes. Et il publie le vent en **km/h**, là où le modèle
commun veut des m/s — le facteur est déclaré dans le mapping et l'unité
réellement annoncée par l'API est vérifiée à chaque run.

In [13]:
from silver.silver_weather import (build_weather, check_timezone, check_units,
                                   explode_hourly, national_average)

raw_meteo = read_source(spark, M, "meteo_archive", "", str(TMP))
check_timezone(raw_meteo, M)          # lève si le lot n'est pas en UTC
alertes = check_units(raw_meteo, M)
print("alertes d'unité :", alertes or "aucune")

weather, w_rejects, w_report = build_weather(explode_hourly(raw_meteo, M), M)
(weather.select("zone_id", F.date_format("ts_utc", "yyyy-MM-dd HH:mm").alias("ts_utc"),
                "temperature_c", "wind_speed_ms", "humidity_pct")
        .orderBy("zone_id", "ts_utc").show(4, truncate=False))
print("36 km/h à la source → 10 m/s en Silver.")

nat = national_average(weather, {"lyon": 0.2, "paris": 0.35},
                       list(M.table("weather").measures))
nat.select(F.date_format("ts_utc", "yyyy-MM-dd HH:mm").alias("ts_utc"),
           F.round("temperature_c", 3).alias("temperature_c"), "zone_id") \
   .orderBy("ts_utc").show(3, truncate=False)
print("moyenne renormalisée sur les villes présentes : (8.0*0.20 + 4.0*0.35)/0.55 "
      f"= {(8.0*0.2 + 4.0*0.35)/0.55:.3f} °C")

2026-08-27 14:36:52,124 INFO    [silver-grid] Lecture meteo_archive (json_arrays) depuis /var/folders/dj/9szbc9yn265_gn9_ktgw85mr0000gn/T/silver_nb_k814vyp6/bronze/meteo_archive/city=*/year=*/month=*/*.json


2026-08-27 14:36:53,201 INFO    [silver-grid] temperature_c : °C -> degC


2026-08-27 14:36:53,219 INFO    [silver-grid] humidity_pct : % -> %


2026-08-27 14:36:53,229 INFO    [silver-grid] wind_speed_ms : km/h -> m/s


2026-08-27 14:36:53,231 INFO    [silver-grid] cloud_cover_pct : % -> %


2026-08-27 14:36:53,254 INFO    [silver-grid] Variables horaires : cloud_cover, relative_humidity_2m, temperature_2m, wind_speed_10m


alertes d'unité : aucune


+-------+----------------+-------------+-------------+------------+
|zone_id|ts_utc          |temperature_c|wind_speed_ms|humidity_pct|
+-------+----------------+-------------+-------------+------------+
|lyon   |2024-03-15 00:00|8.0          |10.0         |80.0        |
|lyon   |2024-03-15 01:00|8.1          |10.0         |80.0        |
|lyon   |2024-03-15 02:00|8.2          |10.0         |80.0        |
|lyon   |2024-03-15 03:00|8.3          |10.0         |80.0        |
+-------+----------------+-------------+-------------+------------+
only showing top 4 rows

36 km/h à la source → 10 m/s en Silver.


+----------------+-------------+-------+
|ts_utc          |temperature_c|zone_id|
+----------------+-------------+-------+
|2024-03-15 00:00|5.455        |fr     |
|2024-03-15 01:00|5.555        |fr     |
|2024-03-15 02:00|5.655        |fr     |
+----------------+-------------+-------+
only showing top 3 rows

moyenne renormalisée sur les villes présentes : (8.0*0.20 + 4.0*0.35)/0.55 = 5.455 °C


## 5. Idempotence de l'écriture

Chaque job reçoit sa fenêtre en paramètre et n'écrase que les partitions
correspondantes, grâce à `partitionOverwriteMode=dynamic`. Rejouer le même lot
ne duplique rien — c'est l'exigence centrale du sujet, vérifiée ici sur Silver
comme `scripts/test_idempotence.py` la vérifie sur Bronze.

In [14]:
from silver.transform import add_partitions, write_silver

out = str(TMP / "silver" / "grid_load")
final = add_partitions(load, M)
n1 = write_silver(final, out, "grid_load")
n2 = write_silver(final, out, "grid_load")   # rejeu de la même fenêtre
relu = spark.read.parquet(out).count()

print(f"écriture : {n1} lignes | rejeu : {n2} lignes | relecture : {relu} lignes")
print("partitions :", sorted(p.name for p in pathlib.Path(out).glob("year=*/month=*")))
assert relu == n1, "le rejeu a dupliqué des lignes"
print("rejeu idempotent confirmé.")

2026-08-27 14:37:02,098 INFO    [silver-grid] grid_load : 105 ligne(s) ecrite(s) dans /var/folders/dj/9szbc9yn265_gn9_ktgw85mr0000gn/T/silver_nb_k814vyp6/silver/grid_load


2026-08-27 14:37:03,773 INFO    [silver-grid] grid_load : 105 ligne(s) ecrite(s) dans /var/folders/dj/9szbc9yn265_gn9_ktgw85mr0000gn/T/silver_nb_k814vyp6/silver/grid_load


écriture : 105 lignes | rejeu : 105 lignes | relecture : 105 lignes
partitions : ['month=10', 'month=3']
rejeu idempotent confirmé.


## 6. Où en est le projet

| Élément | État | Vérification |
|---|---|---|
| `conf/silver_mapping.yml` | **fait** — 3 tables, 27 filières, bornes et unités déclarées | cellules 1 à 3 |
| `src/silver/validation.py` | **fait** — 4 motifs de rejet, quarantaine `_rejects` | 4.2 |
| `src/silver/transform.py` | **fait** — UTC, qualité, dédup fusionnante, dépivotement | 4.1, 4.3, 4.4 |
| `src/silver/readers.py` | **fait** — csv, json_envelope, json_arrays | 4.1 |
| `src/silver/silver_grid.py`, `silver_weather.py` | **fait** — jobs assemblés, rapport JSON | 4.5, 5 |
| `scripts/test_silver_local.py` | **fait** — 33 contrôles sans cluster | `python scripts/test_silver_local.py` |
| `src/gold/`, `dags/`, `src/ml/`, `notebooks/insights.ipynb` | déjà présents | — |

### Points à défendre en soutenance

1. **Le mapping est le contrat.** Aucun nom de champ source dans le Python.
   Une filière ajoutée par RTE se règle dans un YAML.
2. **Rien ne disparaît en silence.** Quatre motifs de rejet, quarantaine avec
   charge utile complète, et un garde‑fou qui fait échouer le job au‑delà de
   25 % de rejets plutôt que de publier une table amputée.
3. **La dédup fusionne, elle ne choisit pas.** Parce que définitif et temps
   réel sont complémentaires, pas redondants.
4. **Le dépivotement connaît la hiérarchie.** `eolien` contient déjà
   `eolien_terrestre + eolien_offshore` : le double compte est évité par
   construction.
5. **Les unités sont vérifiées.** Le vent Open‑Meteo est en km/h ; une colonne
   `wind_speed_ms` qui contiendrait des km/h mentirait de 3,6× jusque dans le
   modèle ML.

### Suite

`gold_build.py` peut désormais exploiter `filiere_level` pour sommer la
production sans risque, et `quality_rank` est déjà matérialisé en Silver pour
éviter de refaire le mapping en aval.

In [15]:
spark.stop()
import shutil; shutil.rmtree(TMP, ignore_errors=True)
print("session Spark fermée, Bronze factice supprimé.")

session Spark fermée, Bronze factice supprimé.
